In [1]:
import pandas as pd
import json
import random

In [3]:
df = pd.read_parquet(
    "../../data/processed/classification dataset/classification_dataset_final.parquet"
)

print(df.shape)

(26688, 4)


In [4]:
instruction_templates = [

    "Analyze the following legal case.",

    "Provide a legal analysis of this dispute.",

    "Explain the legal issues involved in this case.",

    "Determine the legal category and reasoning.",

    "Review the following case and provide legal insights."

]

In [5]:
category_issues = {

    "Criminal":[
        "criminal liability",
        "offence",
        "evidence",
        "prosecution"
    ],

    "Constitutional":[
        "fundamental rights",
        "constitutional validity",
        "judicial review",
        "public law"
    ],

    "Property":[
        "ownership",
        "title dispute",
        "land rights",
        "property transfer"
    ],

    "Contract":[
        "agreement",
        "breach of contract",
        "specific performance",
        "obligations"
    ],

    "Civil":[
        "civil dispute",
        "legal rights",
        "civil remedies",
        "procedural issues"
    ],

    "Family":[
        "marriage",
        "inheritance",
        "family rights",
        "succession"
    ],

    "Service":[
        "employment",
        "promotion",
        "government service",
        "service rules"
    ],

    "Company_Commercial":[
        "commercial transaction",
        "business dispute",
        "corporate law",
        "trade issues"
    ]
}

In [6]:
def build_output(category):

    issues = category_issues.get(
        category,
        ["legal dispute"]
    )

    return f"""
Legal Analysis:

This matter primarily concerns {category} law.

Category:
{category}

Key Legal Issues:
- {issues[0]}
- {issues[1]}
- {issues[2]}
- {issues[3]}

Legal Domain:
{category}

Conclusion:
Based on the facts and legal questions involved, this case is best classified under {category} law.
"""

In [7]:
llm_dataset_v3 = []

for _, row in df.iterrows():

    llm_dataset_v3.append(

        {

            "instruction":
                random.choice(
                    instruction_templates
                ),

            "input":
                row["legal_summary"],

            "output":
                build_output(
                    row["category"]
                )

        }

    )

In [8]:
print(
    len(llm_dataset_v3)
)

llm_dataset_v3[0]

26688


{'instruction': 'Review the following case and provide legal insights.',
 'input': ' The petitioner who was detained under the Preventive Detention Act (Act IV of 1950) applied under Art. 32 of the Constitution for a writ of habeas corpus and for his release from detention, on the ground that the said Act contravened the provisions of Arts. 13, 19, 21 and 22 of the Constitu- tion and was consequently ultra rites and that his detention was therefore illegal: Held, per KANIA C.J., PATANJALI SASTRI, MUKHERJEA and DAS JJ. (FAZL ALI and MAHAJAN JJ. dissentinq)--that the preventive Detention Act, 1950, with the exception of Sec. 14 thereof did not contravene any of the Articles of the Constitution and even though Sec. 14 was ultra rites inas- much as it contravened the provisions of Art. 9.9, (5) of A.K. Gopalan vs The State Of Madras.Union Of India: ... on 19 May, 1950 / 3 the Constitution, as this section was severable from the remaining sections of the Act, the invalidity of Sec. 14 did n

In [9]:
with open(

    "../../data/processed/legal_llm_dataset_v3.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        llm_dataset_v3,

        f,

        ensure_ascii=False,

        indent=2

    )

In [10]:
import os

size_mb = (

    os.path.getsize(
        "../../data/processed/legal_llm_dataset_v3.json"
    )

    /1024

    /1024

)

print(
    round(size_mb,2),
    "MB"
)

56.98 MB


In [11]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map="cpu",
    torch_dtype="auto",
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(
    base_model,
    "models/legal_llm/qwen_legal_v2/checkpoint-2500"
)

tokenizer = AutoTokenizer.from_pretrained(
    "models/legal_llm/qwen_legal_v2/checkpoint-2500"
)

c:\Users\arote\anaconda3\envs\animal_fixed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.67it/s]


ValueError: Can't find 'adapter_config.json' at 'models/legal_llm/qwen_legal_v2/checkpoint-2500'

In [12]:
import os

path = os.path.abspath(
    "models/legal_llm/qwen_legal_v2/checkpoint-2500"
)

print(path)
print(os.path.exists(path))

s:\nyay sarthi\notebooks\legal_llm\models\legal_llm\qwen_legal_v2\checkpoint-2500
False


In [15]:
import os

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from peft import PeftModel

adapter_path = os.path.abspath(
    "../../models/legal_llm/qwen_legal_v2/checkpoint-2500"
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map="cpu",
    torch_dtype="auto",
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

tokenizer = AutoTokenizer.from_pretrained(
    adapter_path
)

print("MODEL LOADED")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


OSError: The paging file is too small for this operation to complete. (os error 1455)